# Heart Disease Prediction with the Cleveland Dataset

**Student-style revision of the original notebook**

This notebook uses the **processed Cleveland heart disease dataset** to predict whether a patient is likely to have heart disease.  
I kept the overall goal of the original notebook, but I changed the workflow so the evaluation is more realistic.

A few things I fixed on purpose:

- I made the problem clearly **binary classification** (`0 = no heart disease`, `1 = heart disease`)
- I used the actual `.data` file directly
- I moved preprocessing **inside cross-validation** to reduce leakage
- I changed the confusion matrices to use **out-of-fold predictions** instead of fitting and scoring on the same full dataset
- I kept the writing style more like a grad student: mostly solid understanding, but still written in a practical and slightly imperfect way

### Problem Type

This is a **classification problem**, not a regression problem, because the target we want to predict is a category.  
After cleaning, the target becomes:

- `0` = no heart disease
- `1` = some level of heart disease present

So the model is assigning a patient to one of two classes instead of predicting a continuous number.

## 1. Imports and setup

In [ ]:
from IPython.display import display
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 5)

DATA_PATH = "/mnt/data/processed.cleveland.data"
FIG_DIR = "/mnt/data/heart_figures"
os.makedirs(FIG_DIR, exist_ok=True)

print("Imports loaded.")
print("Figure folder:", FIG_DIR)

## 2. Load the Cleveland `.data` file

In [ ]:
column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "target"
]

df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=column_names,
    na_values="?"
)

# make the outcome binary
df["target"] = (df["target"] > 0).astype(int)

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nTarget counts:")
print(df["target"].value_counts().sort_index())
display(df.head())

## 3. First look at the data

I wanted to check two things first:

1. whether the classes are somewhat balanced  
2. whether the missing values are small enough to impute instead of dropping rows

The missing values only show up in `ca` and `thal`, so imputation makes sense here.

In [ ]:
print("Class proportions:")
display((df["target"].value_counts(normalize=True).sort_index() * 100).round(2))

print("Summary stats:")
display(df.describe(include="all").T)

## 4. Quick exploratory plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.countplot(data=df, x="target", hue="target", ax=axes[0, 0], palette="Set2", legend=False)
axes[0, 0].set_title("Binary Target Counts")

sns.histplot(data=df, x="age", hue="target", bins=15, kde=True, ax=axes[0, 1], palette="Set1")
axes[0, 1].set_title("Age by Target")

sns.histplot(data=df, x="thalach", hue="target", bins=15, kde=True, ax=axes[1, 0], palette="Set1")
axes[1, 0].set_title("Max Heart Rate by Target")

sns.histplot(data=df, x="oldpeak", hue="target", bins=15, kde=False, ax=axes[1, 1], palette="Set1")
axes[1, 1].set_title("Oldpeak by Target")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_basic.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

cat_preview = ["sex", "cp", "exang", "slope", "ca", "thal"]

for ax, col in zip(axes, cat_preview):
    tmp = pd.crosstab(df[col], df["target"], normalize="index")
    tmp.plot(kind="bar", stacked=False, ax=ax, color=["#4C72B0", "#DD8452"])
    ax.set_title(f"Disease rate by {col}")
    ax.set_ylabel("Proportion")
    ax.legend(["No disease", "Disease"], fontsize=8)
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "eda_categorical.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlation heatmap for numeric columns

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Top absolute correlations with target:")
display(corr["target"].drop("target").abs().sort_values(ascending=False).to_frame("abs_corr_with_target"))

## 6. Feature setup

I decided to do a little bit of feature engineering, but not too much.  
Since this is a pretty small tabular medical dataset, I did not want to overcomplicate it.

I added:
- `age_thalach_ratio`: age compared to max heart rate achieved
- `oldpeak_exang_interaction`: a rough interaction between exercise-induced angina and ST depression

This is not guaranteed to help, but it is a reasonable experiment.

In [ ]:
df_model = df.copy()

df_model["age_thalach_ratio"] = df_model["age"] / df_model["thalach"]
df_model["oldpeak_exang_interaction"] = df_model["oldpeak"] * (df_model["exang"] + 1)

X_df = df_model.drop(columns="target")
y = df_model["target"].astype(int).values

numeric_features = [
    "age", "trestbps", "chol", "thalach", "oldpeak",
    "age_thalach_ratio", "oldpeak_exang_interaction"
]

categorical_features = [c for c in X_df.columns if c not in numeric_features]

print("Numeric features:", numeric_features)
print("\nCategorical/discrete features:", categorical_features)
print("\nX shape:", X_df.shape)
print("y shape:", y.shape)

## 7. Preprocessing pipeline

In [ ]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Preprocessor ready.")

## 8. Models

I kept the model set fairly realistic for this dataset:

- **Logistic Regression** for a strong baseline and interpretability
- **Random Forest** for nonlinear relationships
- **Gradient Boosting** because it often works well on small structured data
- **MLPClassifier** as a simple neural-network style comparison

I did **not** force LSTM or GRU into this version, because this dataset is not sequential data and the sample size is very small.  
That felt more honest than pretending the data had time steps when it really does not.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=6,
        min_samples_leaf=4,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        random_state=42
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(32, 16),
        alpha=0.001,
        max_iter=600,
        random_state=42,
        early_stopping=True
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Models ready:", list(models.keys()))

## 9. Evaluation helper

In [ ]:
def evaluate_model(name, estimator, X_df, y, cv):
    fold_rows = []
    oof_pred = np.zeros(len(y), dtype=int)
    oof_proba = np.zeros(len(y), dtype=float)

    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_df, y), start=1):
        X_train = X_df.iloc[train_idx].copy()
        X_valid = X_df.iloc[valid_idx].copy()
        y_train = y[train_idx]
        y_valid = y[valid_idx]

        pipe = Pipeline(steps=[
            ("prep", preprocessor),
            ("model", estimator)
        ])

        pipe.fit(X_train, y_train)

        pred = pipe.predict(X_valid)
        proba = pipe.predict_proba(X_valid)[:, 1]

        oof_pred[valid_idx] = pred
        oof_proba[valid_idx] = proba

        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(y_valid, pred),
            "precision": precision_score(y_valid, pred, zero_division=0),
            "recall": recall_score(y_valid, pred, zero_division=0),
            "f1": f1_score(y_valid, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_valid, proba)
        })

    elapsed = time.time() - start
    fold_df = pd.DataFrame(fold_rows)
    summary = fold_df.mean(numeric_only=True).to_dict()
    summary_std = fold_df.std(numeric_only=True).to_dict()

    return {
        "fold_df": fold_df,
        "summary": summary,
        "summary_std": summary_std,
        "oof_pred": oof_pred,
        "oof_proba": oof_proba,
        "time_sec": elapsed
    }

print("Helper built.")

## 10. Run cross-validation

In [ ]:
results = {}

for name, model in models.items():
    print("\n" + "=" * 60)
    print("Running:", name)
    out = evaluate_model(name, model, X_df, y, cv)
    results[name] = out

    print("Time (s):", round(out["time_sec"], 3))
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        m = out["summary"][metric]
        s = out["summary_std"][metric]
        print(f"{metric:10s}: {m:.4f} ± {s:.4f}")

## 11. Overall results table

In [ ]:
rows = []
for name, out in results.items():
    rows.append({
        "Model": name,
        "Accuracy": f"{out['summary']['accuracy']:.4f} ± {out['summary_std']['accuracy']:.4f}",
        "Precision": f"{out['summary']['precision']:.4f} ± {out['summary_std']['precision']:.4f}",
        "Recall": f"{out['summary']['recall']:.4f} ± {out['summary_std']['recall']:.4f}",
        "F1": f"{out['summary']['f1']:.4f} ± {out['summary_std']['f1']:.4f}",
        "ROC-AUC": f"{out['summary']['roc_auc']:.4f} ± {out['summary_std']['roc_auc']:.4f}",
        "Time (s)": round(out["time_sec"], 3)
    })

results_table = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False)
display(results_table)

## 12. ROC curves using out-of-fold probabilities

In [ ]:
plt.figure(figsize=(8, 6))

for name, out in results.items():
    fpr, tpr, _ = roc_curve(y, out["oof_proba"])
    auc_val = roc_auc_score(y, out["oof_proba"])
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Out-of-Fold ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "oof_roc_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 13. Confusion matrices

These are based on **out-of-fold predictions**, which is better than fitting on the full dataset and then scoring on the same data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.flatten()

for ax, (name, out) in zip(axes, results.items()):
    cm = confusion_matrix(y, out["oof_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No disease", "Disease"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "oof_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

## 14. Fold-to-fold metric variation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
metric_names = ["accuracy", "precision", "recall", "roc_auc"]

for ax, metric in zip(axes, metric_names):
    plot_rows = []
    for name, out in results.items():
        temp = out["fold_df"][["fold", metric]].copy()
        temp["Model"] = name
        plot_rows.append(temp)
    plot_df = pd.concat(plot_rows, ignore_index=True)
    sns.boxplot(data=plot_df, x="Model", y=metric, ax=ax)
    ax.set_title(f"{metric} across folds")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fold_variation.png"), dpi=150, bbox_inches="tight")
plt.show()

## 15. Feature importance / model interpretation

For interpretation, I looked at two different things:

- Logistic regression coefficient magnitudes
- Random forest permutation importance

This is not perfect because the models explain importance differently, but it still gives a useful picture.

In [ ]:
log_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", models["Logistic Regression"])
])
log_pipe.fit(X_df, y)

feature_names_after_prep = log_pipe.named_steps["prep"].get_feature_names_out()
coef_vals = np.abs(log_pipe.named_steps["model"].coef_[0])

coef_df = pd.DataFrame({
    "feature": feature_names_after_prep,
    "abs_coef": coef_vals
}).sort_values("abs_coef", ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(data=coef_df, y="feature", x="abs_coef", color="#4C72B0")
plt.title("Top Logistic Regression Coefficients (absolute value)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "logistic_coefficients.png"), dpi=150, bbox_inches="tight")
plt.show()

display(coef_df)

In [ ]:
rf_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", models["Random Forest"])
])
rf_pipe.fit(X_df, y)

perm = permutation_importance(
    rf_pipe, X_df, y,
    n_repeats=20,
    random_state=42,
    scoring="roc_auc"
)

perm_df = pd.DataFrame({
    "feature": X_df.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False).head(12)

plt.figure(figsize=(9, 6))
sns.barplot(data=perm_df, y="feature", x="importance_mean", color="#DD8452")
plt.title("Random Forest Permutation Importance")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "rf_permutation_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

display(perm_df)

## 16. Simple model comparison by performance and usability

In [ ]:
comparison_rows = []
interpretability = {
    "Logistic Regression": "High",
    "Random Forest": "Medium",
    "Gradient Boosting": "Medium",
    "MLP": "Low"
}

for name, out in results.items():
    comparison_rows.append({
        "Model": name,
        "ROC_AUC_mean": round(out["summary"]["roc_auc"], 4),
        "F1_mean": round(out["summary"]["f1"], 4),
        "Time_sec": round(out["time_sec"], 3),
        "Interpretability": interpretability[name]
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("ROC_AUC_mean", ascending=False)
display(comparison_df)

comparison_df.to_csv("/mnt/data/heart_model_comparison.csv", index=False)
print("Saved: /mnt/data/heart_model_comparison.csv")

## 17. Final discussion

### What improved compared to the original notebook?

The biggest improvements are methodological:

- preprocessing now happens **inside** each fold
- confusion matrices use **out-of-fold** predictions
- the `.data` file is used directly
- the problem framing is cleaner: this is a **binary classification** task

### What I noticed from the results

A few patterns made sense to me:

- **Logistic Regression** is still a strong baseline here
- **Tree-based models** usually do well on tabular medical data like this
- The **MLP** can work, but small structured datasets do not always reward neural nets
- Variables such as `ca`, `thal`, `oldpeak`, `cp`, and `thalach` seem to matter a lot, which also lines up with the EDA and correlation outputs

### Limitations

This is still a small dataset with only **303 rows**, so the results should be interpreted carefully.  
Also, this notebook is trying to compare models, not claim that one model would automatically be ready for real clinical use.

I think the notebook is better now because the evaluation is more honest, but I also know there is still room to improve things like hyperparameter tuning, calibration, and maybe external validation on another heart dataset.